# Parameter Identification Problem from DA Book

Determine the parameters $b$ and $c$ in

$$ -b u'' + cu' = f, 0<x<1 $$
$$ u(0)=0, u(1) = 0     $$

Let's choose $f = x^2 (1-x)^2$. Now. let's assume we have $u^{\text{obs}}$, we want to find $b$ and $c$ that gives us that solution. A suitable objective function is

$$ J(b, c) = \frac{1}{2} \int_0^1 \left(u - u^{\text{obs}}\right)^2 dx $$

For real world data, a delta function would be required here to only integrate across the points of interest. We need to find the gradient for $J(b , c)$. Doing calculus of variations one gets,

$$ \frac{\delta J}{\delta \alpha} = \int_0^1 \left(u - u^{\text{obs}}\right) \frac{\delta u}{\delta \alpha} dx $$

where $ \alpha \delta b $ and $\alpha \delta c$ is the perturbation of the parameters. Now, let 

$$ \frac{\delta u}{\delta \alpha} = \hat{u} = \lim_{\alpha -> 0} \frac{\tilde{u} - u}{\alpha} $$

where $\tilde{u}$ is the solution at the perturbed parameter values. To get the correct gradient for $J$ we need to solve for $\hat{u}$. Using the forward probem and subsituting for $\tilde{u}$ and $u$ and then subtracting we get,

$$ -b \hat{u}'' + c\hat{u}' = (\delta b) u'' - (\delta c) u' $$

The next step is to multiply this equation by $p$ and integrate, $p$ can then be chosed to satisfy the adjoint problem. This gives,

$$\int_0^1 \hat{u}'' p dx = \int_0^1 \hat{u} p'' dx$$

if $p$ is zero in the boundaries (the adjoint as the same boundary conditions). Also,

$$\int_0^1 \hat{u}'p dx = -\int_0^1 \hat{u} p' dx$$

So, $p$ satisfies the adjoint model, which is

$$ -b p'' - c p' = u - u^{\text{obs}} $$

and $p$ is zero on the boundaries. Now, using this we get 

$$ \nabla_b J = \int_0^1 p u'' $$

and

$$ \nabla_c J = -\int_0^1 p u' $$

So, solving for $u$ and solving for $p$ gives the gradient to minimize the objective function.

In [74]:
from fipy import DiffusionTerm, ConvectionTerm, Grid1D, CellVariable, TransientTerm, Variable, Viewer
import numpy as np
from scipy.optimize import minimize

In [123]:
def forward_problem(b, c, nx):
    mesh = Grid1D(nx=nx, dx=1. / nx)
    X, = mesh.faceCenters
    x, = mesh.cellCenters
    u = CellVariable(mesh=mesh, value=0.)
    cc = CellVariable(mesh=mesh, rank=1)
    cc[0] = c
    u.constrain(0., where=X==0)
    u.constrain(0., where=X==1)
    f = x**2 * (1 - x)**2
    source = CellVariable(mesh=mesh, value=f)
    eqn = TransientTerm() + ConvectionTerm(cc) == DiffusionTerm(b) + source
    eqn.solve(u, dt=1e+10)
    return u

def adjoint_problem(b, c, u, u_obs):
    nx = len(u)
    mesh = Grid1D(nx=nx, dx=1. / nx)
    X, = mesh.faceCenters
    x, = mesh.cellCenters
    p = CellVariable(mesh=mesh, value=0.)
    cc = CellVariable(mesh=mesh, rank=1)
    cc[0] = -c
    p.constrain(0., where=X==0)
    p.constrain(0., where=X==1)
    source = CellVariable(mesh=mesh, value=np.array(u - u_obs))
    eqn = TransientTerm() + ConvectionTerm(cc) == DiffusionTerm(b) + source
    eqn.solve(p, dt=1e+10)
    return p                   

def integral(value):
    return np.sum(1. / len(value) * np.array(value))

def objective_function(params, u_obs):
    nx = len(u_obs)
    b, c = params
    u = forward_problem(b, c, nx)
    return 0.5 * integral((u - u_obs)**2)

def gradient_function(params, u_obs):
    nx = len(u_obs)
    b, c = params
    u = forward_problem(b, c, nx)
    p = adjoint_problem(b, c, u, u_obs)
    u_grad = u.grad[0]
    u_grad_grad = u_grad.grad[0]
    grad_b = integral(p * u_grad_grad)
    grad_c = -integral(p * u_grad)
    return np.array((grad_b, grad_c))

In [128]:
b_obs, c_obs = 1.0, 10.0
nx = 100
b_ini, c_ini = 2.0, 5.0
u_obs = forward_problem(b_obs, c_obs, nx)
u = forward_problem(b_ini, c_ini, nx)

out = minimize(
    objective_function,
    (b_ini, c_ini),
    args=(u_obs,),
    method="BFGS",
    jac=gradient_function,
    tol=1e-9
)    

print(out)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.1107784425503949e-11
        x: [ 1.001e+00  9.968e+00]
      nit: 21
      jac: [-5.768e-10 -7.249e-10]
 hess_inv: [[ 2.341e+06 -2.263e+06]
            [-2.263e+06  3.439e+07]]
     nfev: 41
     njev: 41
